**4. Redes Neuronales**
---
Propiedad de René Adarme Amado
---
Universidad Antonio Nariño - Tesis de Maestría en Hidrogeología Ambiental
Abril de 2025

# **Librerías**

In [13]:
# Importar librerías
from IPython import get_ipython
from IPython.display import display
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score, recall_score,
                             precision_score, roc_auc_score, make_scorer)
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# **Preprocesado**

In [14]:
# Cargar y limpiar la base de datos
df = pd.read_excel("Base_2025_v3.xlsx")
df = df.drop(columns=["ID", "Tipo_Estructuras", "Alturas"])
df["Agua"] = df["Agua"].astype(int)
cat_vars = ["Dureza_Lito", "Num_Estratos", "Clase_Textural"]
for col in cat_vars:
    df[col] = LabelEncoder().fit_transform(df[col])

# Preprocesado de la base de datos
X = df.drop(columns=["Agua"])
y = df["Agua"]

# Estandarizar datos (se hará dentro de la validación cruzada)
scaler = StandardScaler()

# Definir las métricas de evaluación
def especificidad_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        return cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0
    return 0

metricas = {
    'Exactitud': make_scorer(accuracy_score),
    'Sensibilidad': make_scorer(recall_score),
    'Especificidad': make_scorer(especificidad_score),
    'Precision': 'precision',
    'AUC-ROC': 'roc_auc',
    'Puntaje F1': 'f1'
}


#**MODELO 1**

# **Buscar k óptimo**

In [15]:
# 4. Definir modelo base
modelo1 = MLPClassifier(hidden_layer_sizes=(17,), max_iter=2000, random_state=42,
                      alpha=0.1, solver='lbfgs', activation='logistic')

valores_k = list(range(3, 13))
resultados_k = {}

for k in valores_k:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)  # Usar StratifiedKFold
    resultados_cv = cross_validate(modelo1, X, y, cv=skf, scoring=metricas, return_train_score=True)
    resultados_k[k] = {
        metrica: {
            'media_validacion': np.mean(resultados_cv['test_' + metrica]),
            'desviacion': np.std(resultados_cv['test_' + metrica])
        } for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']
    }

# Seleccionar el mejor k
mejor_k = sorted(resultados_k.items(), key=lambda x: (-x[1]['Precision']['media_validacion'], x[1]['Precision']['desviacion'],
                                                      -x[1]['Sensibilidad']['media_validacion'], x[1]['Sensibilidad']['desviacion']))[0][0]

print(f"\nValor óptimo de k para Modelo 1: {mejor_k}")


Valor óptimo de k para Modelo 1: 6


Criterios para escoger k óptimo:
*   Primero: Mayor precisión promedio (orden descendente)
*   Segundo: Menor desviación de exactitud
*   Tercero: Mayor sensibilidad promedio
*   Cuarto: Menor desviación de sensibilidad





# **Modelar con k óptimo**

In [16]:
# 5. Entrenar Modelo 1 con k óptimo
skf = StratifiedKFold(n_splits=mejor_k, shuffle=True, random_state=42)
resultados_modelo1 = cross_validate(modelo1, X, y, cv=skf, scoring=metricas, return_train_score=True)

print("\nResumen de Modelo 1 con el k óptimo en ambos conjuntos de datos:")
for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
    media_train = np.mean(resultados_modelo1['train_' + metrica])
    std_train = np.std(resultados_modelo1['train_' + metrica])
    media_test = np.mean(resultados_modelo1['test_' + metrica])
    std_test = np.std(resultados_modelo1['test_' + metrica])
    print(f"{metrica:<15} | Entrenamiento: {media_train:.4f} ± {std_train:.4f} | Validación: {media_test:.4f} ± {std_test:.4f}")


Resumen de Modelo 1 con el k óptimo en ambos conjuntos de datos:
Precision       | Entrenamiento: 0.8309 ± 0.0260 | Validación: 0.7406 ± 0.0145
Sensibilidad    | Entrenamiento: 0.9167 ± 0.0293 | Validación: 0.8091 ± 0.0426
Puntaje F1      | Entrenamiento: 0.8712 ± 0.0194 | Validación: 0.7727 ± 0.0206
AUC-ROC         | Entrenamiento: 0.8830 ± 0.0350 | Validación: 0.6140 ± 0.0572


# **MODELO 2**

In [19]:
# Modelos 2
def entrenar_y_evaluar_modelo2(X, y, k): # Acepta X, y, k como argumentos
    param_grid2 = {
        'hidden_layer_sizes': [(i,) for i in range(1, 10)],
        'activation': ['tanh', 'relu', 'logistic'],
        'solver': ['sgd', 'adam', 'lbfgs'],
        'alpha': [0.01],
        'learning_rate': ['constant', 'adaptive']
    }

    gs2 = GridSearchCV(MLPClassifier(max_iter=500, random_state=42),
                       param_grid2, cv=k, scoring='f1', verbose=0, n_jobs=-1) # Usa el k óptimo
    gs2.fit(X, y)

    print("\nMejores hiperparámetros para el modelo 2:", gs2.best_params_)

    modelo2 = MLPClassifier(**gs2.best_params_, max_iter=2000, random_state=42)
    resultados = cross_validate(modelo2, X, y, cv=k, scoring=metricas, return_train_score=True) # Usa el k óptimo

    print("\nResumen de métricas para Modelo 2 (k = {}):\n".format(k))
    print("{:<15}{:<15}{:<10}{}".format("Métrica", "Conjunto", "Media", "Desviación"))
    for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
        media_train = np.mean(resultados['train_' + metrica])
        std_train = np.std(resultados['train_' + metrica])
        media_val = np.mean(resultados['test_' + metrica])
        std_val = np.std(resultados['test_' + metrica])
        print("{:<15}{:<15}{:.4f}   {:.4f}".format(metrica, "Entrenamiento", media_train, std_train))
        print("{:<15}{:<15}{:.4f}   {:.4f}".format(metrica, "Validación", media_val, std_val))

    return gs2.best_estimator_, resultados

# Guardar los resultados
modelo2, resultados_modelo2 = entrenar_y_evaluar_modelo2(X, y, mejor_k)


Mejores hiperparámetros para el modelo 2: {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (1,), 'learning_rate': 'constant', 'solver': 'lbfgs'}

Resumen de métricas para Modelo 2 (k = 6):

Métrica        Conjunto       Media     Desviación
Precision      Entrenamiento  0.7003   0.0136
Precision      Validación     0.7049   0.0232
Sensibilidad   Entrenamiento  0.9830   0.0380
Sensibilidad   Validación     0.9974   0.0059
Puntaje F1     Entrenamiento  0.8172   0.0052
Puntaje F1     Validación     0.8257   0.0135
AUC-ROC        Entrenamiento  0.5517   0.0784
AUC-ROC        Validación     0.5101   0.0197


# **MODELO 3**

In [20]:
# Modelos 3
def entrenar_y_evaluar_modelo3(X, y, k):
    param_grid3 = {
        'hidden_layer_sizes': [(21, i) for i in range(18, 25)],
        'activation': ['relu', 'logistic'],
        'solver': ['sgd', 'lbfgs'],
        'alpha': [0.01],
        'learning_rate': ['constant', 'adaptive']
    }

    gs3 = GridSearchCV(MLPClassifier(max_iter=500, random_state=42),
                       param_grid3, cv=k, scoring='f1', verbose=0, n_jobs=-1) # Usa el k óptimo
    gs3.fit(X, y)

    mejores_parametros3 = gs3.best_params_
    print("\nMejores hiperparámetros para el Modelo 3:", mejores_parametros3)

    modelo3 = MLPClassifier(**mejores_parametros3, max_iter=5000, random_state=42)
    resultados = cross_validate(modelo3, X, y, cv=k, scoring=metricas, return_train_score=True) # Usa el k óptimo

    print("\nResumen de métricas para Modelo 3 (k = {}):\n".format(k))
    print("{:<15}{:<15}{:<10}{}".format("Métrica", "Conjunto", "Media", "Desviación"))
    for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
        media_train = np.mean(resultados['train_' + metrica])
        std_train = np.std(resultados['train_' + metrica])
        media_val = np.mean(resultados['test_' + metrica])
        std_val = np.std(resultados['test_' + metrica])
        print("{:<15}{:<15}{:.4f}   {:.4f}".format(metrica, "Entrenamiento", media_train, std_train))
        print("{:<15}{:<15}{:.4f}   {:.4f}".format(metrica, "Validación", media_val, std_val))

    return gs3.best_estimator_, resultados

# Guardar los resultados
modelo3, resultados_modelo3 = entrenar_y_evaluar_modelo3(X, y, mejor_k)


Mejores hiperparámetros para el Modelo 3: {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (21, 23), 'learning_rate': 'constant', 'solver': 'sgd'}

Resumen de métricas para Modelo 3 (k = 6):

Métrica        Conjunto       Media     Desviación
Precision      Entrenamiento  0.6998   0.0096
Precision      Validación     0.7026   0.0188
Sensibilidad   Entrenamiento  0.9878   0.0200
Sensibilidad   Validación     0.9947   0.0075
Puntaje F1     Entrenamiento  0.8190   0.0016
Puntaje F1     Validación     0.8234   0.0144
AUC-ROC        Entrenamiento  0.6758   0.0496
AUC-ROC        Validación     0.5828   0.1455


# **MODELO 4**

In [21]:
 # Modelo 4
def entrenar_y_evaluar_modelo4(X, y, k):
    param_grid4 = {
        'hidden_layer_sizes': [(i, j) for i in range(1, 10) for j in range(1, 10)],
        'activation': ['tanh', 'relu', 'logistic'],
        'solver': ['sgd', 'adam', 'lbfgs'],
        'alpha': [0.01, 0.001],
        'learning_rate': ['constant', 'adaptive']
    }

    gs4 = GridSearchCV(MLPClassifier(max_iter=500, random_state=42),
                       param_grid4, cv=k, scoring='roc_auc', verbose=0, n_jobs=-1) # Usa el k óptimo
    gs4.fit(X, y)

    mejores_parametros4 = gs4.best_params_
    print("\nMejores hiperparámetros para el Modelo 4:", mejores_parametros4)

    modelo4 = MLPClassifier(**mejores_parametros4, max_iter=5000, random_state=42)
    resultados = cross_validate(modelo4, X, y, cv=k, scoring=metricas, return_train_score=True) # Usa el k óptimo

    print("\nResumen de métricas para Modelo 4 (k = {}):\n".format(k))
    print("{:<15}{:<15}{:<10}{}".format("Métrica", "Conjunto", "Media", "Desviación"))
    for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
        media_train = np.mean(resultados['train_' + metrica])
        std_train = np.std(resultados['train_' + metrica])
        media_val = np.mean(resultados['test_' + metrica])
        std_val = np.std(resultados['test_' + metrica])
        print("{:<15}{:<15}{:.4f}   {:.4f}".format(metrica, "Entrenamiento", media_train, std_train))
        print("{:<15}{:<15}{:.4f}   {:.4f}".format(metrica, "Validación", media_val, std_val))

    return gs4.best_estimator_, resultados

# Guardar los resultados
modelo4, resultados_modelo4 = entrenar_y_evaluar_modelo4(X, y, mejor_k)


Mejores hiperparámetros para el Modelo 4: {'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (8, 8), 'learning_rate': 'adaptive', 'solver': 'sgd'}

Resumen de métricas para Modelo 4 (k = 6):

Métrica        Conjunto       Media     Desviación
Precision      Entrenamiento  0.7115   0.0171
Precision      Validación     0.6939   0.0045
Sensibilidad   Entrenamiento  0.9486   0.0488
Sensibilidad   Validación     0.9150   0.1334
Puntaje F1     Entrenamiento  0.8121   0.0102
Puntaje F1     Validación     0.7847   0.0581
AUC-ROC        Entrenamiento  0.6631   0.0454
AUC-ROC        Validación     0.6342   0.2108


# **Resumen de modelos**

In [22]:
print("\nResumen de Modelos:\n")
print("{:<10} {:<15} {:<20} {:<20}".format("Modelo", "Métrica", "Entrenamiento", "Validación"))
print("-" * 65)

# Cambio: Usar las variables resultados_modelo2, resultados_modelo3 y resultados_modelo4
resultados_modelos = [resultados_modelo1, resultados_modelo2, resultados_modelo3, resultados_modelo4]

for i, resultados in enumerate(resultados_modelos):
    print("Modelo {}".format(i + 1))
    for metrica in ['Precision', 'Sensibilidad', 'Puntaje F1', 'AUC-ROC']:
        media_train = np.mean(resultados['train_' + metrica])
        std_train = np.std(resultados['train_' + metrica])
        media_val = np.mean(resultados['test_' + metrica])
        std_val = np.std(resultados['test_' + metrica])
        print("{:<10} {:<15} {:.4f} ± {:.4f}  {:.4f} ± {:.4f}".format("", metrica, media_train, std_train, media_val, std_val))
    print("-" * 65)


Resumen de Modelos:

Modelo     Métrica         Entrenamiento        Validación          
-----------------------------------------------------------------
Modelo 1
           Precision       0.8309 ± 0.0260  0.7406 ± 0.0145
           Sensibilidad    0.9167 ± 0.0293  0.8091 ± 0.0426
           Puntaje F1      0.8712 ± 0.0194  0.7727 ± 0.0206
           AUC-ROC         0.8830 ± 0.0350  0.6140 ± 0.0572
-----------------------------------------------------------------
Modelo 2
           Precision       0.7003 ± 0.0136  0.7049 ± 0.0232
           Sensibilidad    0.9830 ± 0.0380  0.9974 ± 0.0059
           Puntaje F1      0.8172 ± 0.0052  0.8257 ± 0.0135
           AUC-ROC         0.5517 ± 0.0784  0.5101 ± 0.0197
-----------------------------------------------------------------
Modelo 3
           Precision       0.6998 ± 0.0096  0.7026 ± 0.0188
           Sensibilidad    0.9878 ± 0.0200  0.9947 ± 0.0075
           Puntaje F1      0.8190 ± 0.0016  0.8234 ± 0.0144
           AUC-ROC      